In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
import matplotlib.pyplot as plt


In [51]:
df = pd.read_csv("homework3data.csv")
df.head()

,month,year,nfirms,me,div_yld,log_excess_ret
0,1,1960,1088,2.850929e+08,0.033025,-0.072275
1,2,1960,1092,2.881785e+08,0.033130,0.010698
2,3,1960,1096,2.841433e+08,0.033801,-0.016199
3,4,1960,1096,2.795177e+08,0.034413,-0.018143
4,5,1960,1098,2.877926e+08,0.033933,0.030841


Question 2

In [52]:
df["rt"] = df.log_excess_ret
df["dy"] = df["div_yld"]
var_df = df[["rt", "dy"]].dropna()
model = VAR(var_df)

bic = []
for lag in [1, 2, 3]:
    VAR_lag = model.fit(lag)
    VAR_bic = VAR_lag.bic
    bic.append(float(VAR_bic))
    print("BIC for lag", lag, ":", VAR_bic)
print("\n", bic)

chosen_lag = bic.index(min(bic)) + 1
print("\nChosen lag:", chosen_lag)

chosen_VAR = model.fit(chosen_lag)
print("\n", chosen_VAR.summary())

BIC for lag 1 : -20.825254777973466
BIC for lag 2 : -20.788510145916018
BIC for lag 3 : -20.75314568445613

 [-20.825254777973466, -20.788510145916018, -20.75314568445613]

Chosen lag: 1

   Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Wed, 25, Feb, 2026
Time:                     00:43:54
--------------------------------------------------------------------
No. of Equations:         2.00000    BIC:                   -20.8253
Nobs:                     719.000    HQIC:                  -20.8487
Log likelihood:           5465.98    FPE:                8.69192e-10
AIC:                     -20.8635    Det(Omega_mle):     8.61984e-10
--------------------------------------------------------------------
Results for equation rt
           coefficient       std. error           t-stat            prob
------------------------------------------------------------------------
const        -0.003027         0.004978           -0

Question 3

In [79]:

def compute_gamma_var (phi1, sigma):

    phi = np.asarray(phi1, float)
    sigma = np.asarray(sigma, float)

    k = phi.shape[0]
    I = np.eye(k * k)
    kronec = np.kron(phi, phi)

    vecSigma = sigma.reshape(-1, order = "F")

    vecGamma = np.linalg.solve(I - kronec, vecSigma)

    gamma = vecGamma.reshape(k, k, order = "F")

    var_vec = np.diag(gamma)

    return gamma , var_vec

phi = chosen_VAR.coefs[0]
sigma = chosen_VAR.sigma_u

gamma, var_vec = compute_gamma_var(phi, sigma)

var_rt = var_vec[0]
var_dy = var_vec[1]
cov_rt_dy = gamma[0][1]

pop_beta = 12 * cov_rt_dy / (var_dy)

pop_beta


np.float64(-4.314428408533772)

Question 4

Estimate a simple restricted VAR. Project the return on just a constant; project the dividend
yield onto a constant and its own lag. Save these coefficients. For each data point t = 2 through
T, you have a vector of residuals. Compute the correlation matrix of these residual vectors. Are
the residuals correlated?